# Module 05: LLM Hosting & API Exposure

**Companion notebook for**: `05-llm-hosting.html`

## Learning Objectives
- Understand when to self-host LLMs vs. using commercial APIs (cost, latency, compliance)
- Configure and use **vLLM** and **TGI** as production inference engines
- Set up **Ollama** for local development with OpenAI-compatible endpoints
- Build **FastAPI** wrapper services with streaming, health checks, and error handling
- Deploy LLM services using **Docker** containers and AWS **ECS**
- Benchmark inference latency and throughput across different hosting options
- Calculate cost tradeoffs between cloud APIs and self-hosted infrastructure

## Prerequisites
- Ollama installed locally (for Sections 2-3): https://ollama.com
- OpenAI API key set as `OPENAI_API_KEY` environment variable (for cloud comparisons)
- Python 3.10+ with pip

---
## Setup & Dependencies

In [ ]:
%pip install -q openai requests fastapi uvicorn pydantic httpx tabulate

In [ ]:
import os
import json
import time
import statistics
from typing import AsyncGenerator

import requests
import httpx
from tabulate import tabulate

# Optional: OpenAI key for cloud API comparison (not required for local Ollama sections)
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if OPENAI_API_KEY:
    print("OpenAI API key found -- cloud comparison sections will work.")
else:
    print("No OPENAI_API_KEY set -- cloud comparison sections will be skipped.")
    print("Set it with: os.environ['OPENAI_API_KEY'] = 'sk-...'")

print("All imports successful.")

---
## Section 1: Self-Hosting Decision Framework

Before writing any code, the first question is **when does self-hosting make sense?**

The decision depends on five dimensions:

| Dimension | API Wins When... | Self-Host Wins When... |
|-----------|-----------------|------------------------|
| **Cost** | < $5K/month API spend | > $5K/month API spend |
| **Data Governance** | No PII/PHI constraints | HIPAA, SOC 2, GDPR requirements |
| **Latency** | > 200ms TTFT acceptable | < 100ms TTFT needed |
| **Model Customization** | Standard models suffice | Fine-tuned or custom weights |
| **Operational Maturity** | Small team, no ML-ops | Dedicated infra team |

**Rule of thumb**: Start with commercial APIs unless two or more dimensions favor self-hosting.

In [ ]:
# -------------------------------------------------------
# Cost Comparison Calculator: Cloud API vs. Self-Hosted
# -------------------------------------------------------

def calculate_monthly_costs(
    queries_per_day: int,
    avg_input_tokens: int = 2000,
    avg_output_tokens: int = 500,
    # Cloud API pricing (per million tokens)
    api_input_price_per_m: float = 2.50,    # e.g., GPT-4o input
    api_output_price_per_m: float = 10.00,   # e.g., GPT-4o output
    # Self-hosted pricing
    gpu_hourly_cost: float = 1.52,           # g5.2xlarge (1x A10G)
    num_gpus: int = 1,
    engineering_hours_per_month: float = 20,  # Maintenance overhead
    engineering_hourly_rate: float = 100,
):
    """Compare monthly costs of cloud API vs. self-hosted LLM deployment."""
    days_per_month = 30
    total_queries = queries_per_day * days_per_month

    # Cloud API costs
    input_tokens_total = total_queries * avg_input_tokens
    output_tokens_total = total_queries * avg_output_tokens
    api_input_cost = (input_tokens_total / 1_000_000) * api_input_price_per_m
    api_output_cost = (output_tokens_total / 1_000_000) * api_output_price_per_m
    api_total = api_input_cost + api_output_cost

    # Self-hosted costs
    gpu_monthly = gpu_hourly_cost * 24 * days_per_month * num_gpus
    engineering_monthly = engineering_hours_per_month * engineering_hourly_rate
    self_hosted_total = gpu_monthly + engineering_monthly

    savings = api_total - self_hosted_total
    savings_pct = (savings / api_total * 100) if api_total > 0 else 0

    return {
        "queries_per_day": queries_per_day,
        "total_monthly_queries": total_queries,
        "api_cost": round(api_total, 2),
        "api_input_cost": round(api_input_cost, 2),
        "api_output_cost": round(api_output_cost, 2),
        "self_hosted_cost": round(self_hosted_total, 2),
        "gpu_cost": round(gpu_monthly, 2),
        "engineering_cost": round(engineering_monthly, 2),
        "monthly_savings": round(savings, 2),
        "savings_pct": round(savings_pct, 1),
        "recommendation": "Self-Host" if savings > 0 else "Use API",
    }


# Compare across different query volumes
volumes = [1_000, 5_000, 10_000, 50_000, 100_000]
results = [calculate_monthly_costs(v) for v in volumes]

table_data = [
    [
        f"{r['queries_per_day']:,}",
        f"${r['api_cost']:,.0f}",
        f"${r['self_hosted_cost']:,.0f}",
        f"${r['monthly_savings']:+,.0f}",
        r["recommendation"],
    ]
    for r in results
]

print("Monthly Cost Comparison: GPT-4o API vs. Self-Hosted (1x A10G + Llama 3 8B)")
print("=" * 80)
print(
    tabulate(
        table_data,
        headers=["Queries/Day", "API Cost", "Self-Hosted", "Savings", "Recommendation"],
        tablefmt="grid",
    )
)
print("\nAssumptions: 2K input + 500 output tokens/query, g5.2xlarge @ $1.52/hr, 20 hrs/mo eng.")

---
## Section 2: Inference Engines -- vLLM & TGI

Inference engines are the runtime that loads model weights into GPU memory and generates
text efficiently. The two dominant open-source options are:

- **vLLM** (UC Berkeley) -- Uses PagedAttention for optimal GPU memory management;
  exposes an OpenAI-compatible API out of the box.
- **TGI** (Hugging Face) -- Rust + Python core; tight HF Hub integration.

Both support continuous batching, KV-cache management, tensor parallelism, and
quantized models (GPTQ, AWQ).

In [ ]:
# -------------------------------------------------------
# vLLM Server Configuration Examples
# -------------------------------------------------------
# These are command-line examples -- run them in a terminal, not in this notebook.
# We print them here for reference and documentation.

vllm_configs = {
    "Basic 8B Model (single GPU)": {
        "command": (
            "vllm serve meta-llama/Meta-Llama-3-8B-Instruct "
            "--host 0.0.0.0 "
            "--port 8000 "
            "--dtype auto "
            "--max-model-len 8192 "
            "--gpu-memory-utilization 0.90 "
            "--enable-chunked-prefill "
            "--max-num-seqs 256"
        ),
        "gpu": "1x A10G (24 GB) or better",
        "notes": "Good starting point. Chunked prefill reduces time-to-first-token.",
    },
    "70B Model (multi-GPU tensor parallel)": {
        "command": (
            "vllm serve meta-llama/Meta-Llama-3-70B-Instruct "
            "--tensor-parallel-size 2 "
            "--max-model-len 4096 "
            "--gpu-memory-utilization 0.92"
        ),
        "gpu": "2x A100-80GB",
        "notes": "Tensor parallelism shards weights across GPUs automatically.",
    },
    "Quantized 70B (single GPU)": {
        "command": (
            "vllm serve TheBloke/Llama-3-70B-Instruct-GPTQ "
            "--quantization gptq "
            "--max-model-len 4096 "
            "--dtype float16"
        ),
        "gpu": "1x A100-80GB (4-bit quantized fits in ~35 GB)",
        "notes": "GPTQ 4-bit: ~1-3% quality drop, 4x memory savings.",
    },
}

for name, cfg in vllm_configs.items():
    print(f"--- {name} ---")
    print(f"  GPU: {cfg['gpu']}")
    print(f"  Command:")
    print(f"    {cfg['command']}")
    print(f"  Notes: {cfg['notes']}")
    print()

In [ ]:
# -------------------------------------------------------
# Model Serving Comparison Table
# -------------------------------------------------------

comparison_data = [
    ["Language", "Python + CUDA", "Rust + Python + CUDA", "C++ (llama.cpp)"],
    ["API Format", "OpenAI-compatible", "Custom + OpenAI-compatible", "OpenAI-compatible"],
    ["PagedAttention", "Native (invented here)", "Adopted", "No"],
    ["Tensor Parallelism", "Yes (simple flag)", "Yes", "No (single GPU/CPU)"],
    ["Speculative Decoding", "Yes", "Yes (built-in)", "Yes"],
    ["Quantization", "GPTQ, AWQ, FP8, GGUF", "GPTQ, AWQ, EETQ, bnb", "GGUF (all quant levels)"],
    ["Deployment", "pip / Docker", "Docker (primary)", "brew / curl install"],
    ["Best For", "Production throughput", "HF ecosystem", "Local dev / prototyping"],
    ["Target Hardware", "Data center GPUs", "Data center GPUs", "Laptop / consumer GPU"],
]

print("Inference Engine Comparison: vLLM vs. TGI vs. Ollama")
print("=" * 90)
print(
    tabulate(
        comparison_data,
        headers=["Feature", "vLLM", "TGI", "Ollama"],
        tablefmt="grid",
    )
)

---
## Section 3: Ollama for Local Development

Ollama packages model downloading, quantization, and serving into a single CLI tool.
It is ideal for **local development and prototyping** before deploying to production
with vLLM or TGI.

### Quick Start (run in terminal)
```bash
# Install (macOS/Linux)
curl -fsSL https://ollama.com/install.sh | sh

# Pull and run a model
ollama run llama3:8b

# List downloaded models
ollama list
```

Ollama exposes a REST API on `localhost:11434` and an OpenAI-compatible endpoint
at `/v1/chat/completions`.

> **Important**: The cells below require Ollama running locally with a model pulled.
> Run `ollama pull llama3:8b` in a terminal first.

In [ ]:
# -------------------------------------------------------
# Check if Ollama is running and list available models
# -------------------------------------------------------

OLLAMA_BASE_URL = "http://localhost:11434"


def check_ollama_status():
    """Verify Ollama is running and return available models."""
    try:
        resp = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        resp.raise_for_status()
        models = resp.json().get("models", [])
        print(f"Ollama is running at {OLLAMA_BASE_URL}")
        print(f"Available models ({len(models)}):")
        for m in models:
            size_gb = m.get("size", 0) / (1024 ** 3)
            print(f"  - {m['name']}  ({size_gb:.1f} GB)")
        return models
    except requests.ConnectionError:
        print("Ollama is NOT running. Start it with: ollama serve")
        return []
    except Exception as e:
        print(f"Error connecting to Ollama: {e}")
        return []


available_models = check_ollama_status()

In [ ]:
# -------------------------------------------------------
# Ollama Native API: Chat Completion (non-streaming)
# -------------------------------------------------------

def ollama_chat(model: str, messages: list, stream: bool = False) -> dict:
    """Call Ollama's native /api/chat endpoint."""
    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json={
            "model": model,
            "messages": messages,
            "stream": stream,
        },
        timeout=120,
    )
    response.raise_for_status()
    return response.json()


# Pick the first available model, or default to llama3:8b
MODEL = available_models[0]["name"] if available_models else "llama3:8b"

if available_models:
    result = ollama_chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Be concise."},
            {"role": "user", "content": "Explain KV caching in transformers in 2-3 sentences."},
        ],
    )
    print(f"Model: {MODEL}")
    print(f"Response:\n{result['message']['content']}")
    print(f"\nTokens -- prompt: {result.get('prompt_eval_count', 'N/A')}, "
          f"completion: {result.get('eval_count', 'N/A')}")
    print(f"Total duration: {result.get('total_duration', 0) / 1e9:.2f}s")
else:
    print("Skipped: Ollama not available. Pull a model with: ollama pull llama3:8b")

In [ ]:
# -------------------------------------------------------
# Ollama Native API: Streaming Chat
# -------------------------------------------------------

def ollama_chat_stream(model: str, messages: list):
    """Stream tokens from Ollama's /api/chat endpoint."""
    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json={"model": model, "messages": messages, "stream": True},
        stream=True,
        timeout=120,
    )
    response.raise_for_status()

    full_content = ""
    for line in response.iter_lines():
        if line:
            chunk = json.loads(line)
            token = chunk.get("message", {}).get("content", "")
            print(token, end="", flush=True)
            full_content += token
            if chunk.get("done", False):
                break
    print()  # final newline
    return full_content


if available_models:
    print(f"Streaming from {MODEL}:\n")
    _ = ollama_chat_stream(
        model=MODEL,
        messages=[{"role": "user", "content": "Write a haiku about GPU inference."}],
    )
else:
    print("Skipped: Ollama not available.")

### OpenAI-Compatible Endpoint via Ollama

Ollama exposes `/v1/chat/completions` so you can use the **standard OpenAI Python SDK**.
This means your code is portable -- switch from Ollama to OpenAI, vLLM, or any
OpenAI-compatible endpoint by changing a single environment variable.

In [ ]:
# -------------------------------------------------------
# Using OpenAI SDK with Ollama (OpenAI-compatible endpoint)
# -------------------------------------------------------
from openai import OpenAI


def get_openai_compatible_client(base_url: str = "http://localhost:11434/v1"):
    """
    Create an OpenAI client pointed at a local Ollama (or vLLM) server.
    The same client works with OpenAI, vLLM, TGI, or Ollama -- just change base_url.
    """
    return OpenAI(
        base_url=base_url,
        api_key="ollama",  # Ollama does not require a real API key
    )


if available_models:
    client = get_openai_compatible_client()

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a concise technical writer."},
            {"role": "user", "content": "What is continuous batching in LLM serving?"},
        ],
        temperature=0.7,
        max_tokens=300,
    )

    print(f"Model: {response.model}")
    print(f"Response:\n{response.choices[0].message.content}")
    if response.usage:
        print(f"\nUsage: {response.usage.prompt_tokens} prompt + "
              f"{response.usage.completion_tokens} completion = "
              f"{response.usage.total_tokens} total tokens")
else:
    print("Skipped: Ollama not available.")
    print("Tip: To use vLLM instead, change base_url to 'http://localhost:8000/v1'")

### Ollama Modelfile (Custom Model Configuration)

Modelfiles give you Dockerfile-like control over model behavior. You define a base model,
system prompt, and parameter overrides.

In [ ]:
# -------------------------------------------------------
# Example: Ollama Modelfile for a Custom Coding Assistant
# -------------------------------------------------------
# Save this content to a file called 'Modelfile.code-review' and run:
#   ollama create code-reviewer -f Modelfile.code-review
#   ollama run code-reviewer

modelfile_content = '''FROM codellama:13b

SYSTEM """You are an expert code reviewer. Analyze the provided code for:
1. Bugs and potential runtime errors
2. Security vulnerabilities
3. Performance issues
4. Code style and readability
Provide specific, actionable feedback with corrected code examples."""

PARAMETER temperature 0.2
PARAMETER top_p 0.9
PARAMETER num_ctx 8192
PARAMETER stop "</review>"
'''

print("Modelfile.code-review:")
print("=" * 50)
print(modelfile_content)
print("Usage:")
print("  ollama create code-reviewer -f Modelfile.code-review")
print("  ollama run code-reviewer")

---
## Section 4: FastAPI LLM Service Wrapper

Production applications need more than raw inference: input validation, authentication,
rate limiting, prompt templates, logging, and structured error handling.

The architecture is a three-layer sandwich:
1. **Inference engine** (vLLM / Ollama) at the bottom
2. **FastAPI wrapper** in the middle (validation, routing, logging)
3. **Client / frontend** at the top

Below is a complete, runnable FastAPI service definition.

In [ ]:
# -------------------------------------------------------
# FastAPI LLM Service -- Complete Example
# -------------------------------------------------------
# This cell defines the service. To actually run it:
#   1. Save this code to a file called 'llm_service.py'
#   2. Run: uvicorn llm_service:app --host 0.0.0.0 --port 8080

from pydantic import BaseModel, Field


# --- Pydantic Request/Response Models ---

class Message(BaseModel):
    role: str = Field(..., pattern="^(system|user|assistant)$")
    content: str = Field(..., min_length=1, max_length=100_000)


class ChatRequest(BaseModel):
    messages: list[Message] = Field(..., min_length=1)
    model: str = "llama3:8b"
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    max_tokens: int = Field(default=1024, ge=1, le=4096)
    stream: bool = False


class ChatResponse(BaseModel):
    content: str
    model: str
    usage: dict
    latency_ms: float


# --- Validate that the models work ---

sample_request = ChatRequest(
    messages=[
        Message(role="system", content="You are helpful."),
        Message(role="user", content="Hello!"),
    ],
    temperature=0.5,
    max_tokens=512,
)

print("Validated ChatRequest:")
print(json.dumps(sample_request.model_dump(), indent=2))

In [ ]:
# -------------------------------------------------------
# Full FastAPI Application Code (save as llm_service.py)
# -------------------------------------------------------

FASTAPI_APP_CODE = '''
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from openai import AsyncOpenAI
from typing import AsyncGenerator
import json, time, logging, os

logger = logging.getLogger(__name__)
app = FastAPI(title="LLM Service", version="1.0.0")

# --- Pydantic Models ---

class Message(BaseModel):
    role: str = Field(..., pattern="^(system|user|assistant)$")
    content: str = Field(..., min_length=1, max_length=100_000)

class ChatRequest(BaseModel):
    messages: list[Message] = Field(..., min_length=1)
    model: str = "llama3:8b"
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    max_tokens: int = Field(default=1024, ge=1, le=4096)
    stream: bool = False

class ChatResponse(BaseModel):
    content: str
    model: str
    usage: dict
    latency_ms: float

# --- Backend Client (points to vLLM or Ollama) ---

BACKEND_URL = os.environ.get("LLM_BACKEND_URL", "http://localhost:11434/v1")
client = AsyncOpenAI(base_url=BACKEND_URL, api_key="not-needed")

# --- Non-Streaming Endpoint ---

@app.post("/v1/chat", response_model=ChatResponse)
async def chat(req: ChatRequest):
    start = time.perf_counter()
    try:
        response = await client.chat.completions.create(
            model=req.model,
            messages=[m.model_dump() for m in req.messages],
            temperature=req.temperature,
            max_tokens=req.max_tokens,
        )
    except Exception as e:
        logger.error(f"Inference failed: {e}")
        raise HTTPException(status_code=502, detail="Inference backend unavailable")

    elapsed = (time.perf_counter() - start) * 1000
    choice = response.choices[0]
    logger.info(f"model={req.model} tokens={response.usage.total_tokens} latency={elapsed:.0f}ms")

    return ChatResponse(
        content=choice.message.content,
        model=response.model,
        usage={
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens,
        },
        latency_ms=round(elapsed, 2),
    )

# --- Streaming Endpoint ---

async def stream_tokens(req: ChatRequest) -> AsyncGenerator[str, None]:
    """Async generator that yields SSE-formatted token chunks."""
    try:
        stream = await client.chat.completions.create(
            model=req.model,
            messages=[m.model_dump() for m in req.messages],
            temperature=req.temperature,
            max_tokens=req.max_tokens,
            stream=True,
        )
        async for chunk in stream:
            delta = chunk.choices[0].delta.content
            if delta:
                event = json.dumps({"content": delta})
                yield f"data: {event}\\n\\n"
        yield "data: [DONE]\\n\\n"
    except Exception as e:
        error = json.dumps({"error": str(e)})
        yield f"data: {error}\\n\\n"

@app.post("/v1/chat/stream")
async def chat_stream(req: ChatRequest):
    return StreamingResponse(
        stream_tokens(req),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},
    )

# --- Health Checks ---

@app.get("/health")
async def health():
    """Readiness probe: verifies inference backend is responding."""
    try:
        models = await client.models.list()
        return {"status": "healthy", "models": [m.id for m in models.data]}
    except Exception as e:
        raise HTTPException(status_code=503, detail=f"Backend unhealthy: {e}")

@app.get("/live")
async def liveness():
    """Liveness probe: confirms process is alive."""
    return {"status": "alive"}
'''

print("FastAPI LLM Service (save as llm_service.py):")
print("=" * 60)
print(FASTAPI_APP_CODE)
print("\nRun with:")
print("  uvicorn llm_service:app --host 0.0.0.0 --port 8080 --workers 4 --loop uvloop")

---
## Section 5: Health Checks & Monitoring Patterns

Production LLM services need three types of monitoring:
1. **Liveness probe** -- Is the process alive? (simple HTTP 200)
2. **Readiness probe** -- Can it serve traffic? (model loaded and warm)
3. **Performance monitoring** -- Latency percentiles, throughput, error rates

In [ ]:
# -------------------------------------------------------
# Health Check & Monitoring Client
# -------------------------------------------------------

def check_endpoint_health(base_url: str, timeout: float = 5.0) -> dict:
    """
    Check health of an LLM endpoint (Ollama, vLLM, or custom FastAPI).
    Returns status, available models, and response time.
    """
    result = {
        "url": base_url,
        "status": "unknown",
        "response_time_ms": None,
        "models": [],
        "error": None,
    }

    start = time.perf_counter()
    try:
        # Try OpenAI-compatible /v1/models endpoint
        resp = requests.get(f"{base_url}/v1/models", timeout=timeout)
        elapsed_ms = (time.perf_counter() - start) * 1000

        if resp.status_code == 200:
            data = resp.json()
            models = [m.get("id", "unknown") for m in data.get("data", [])]
            result.update({
                "status": "healthy",
                "response_time_ms": round(elapsed_ms, 1),
                "models": models,
            })
        else:
            result.update({
                "status": "degraded",
                "response_time_ms": round(elapsed_ms, 1),
                "error": f"HTTP {resp.status_code}",
            })
    except requests.ConnectionError:
        result.update({"status": "unreachable", "error": "Connection refused"})
    except requests.Timeout:
        result.update({"status": "timeout", "error": f"Timed out after {timeout}s"})
    except Exception as e:
        result.update({"status": "error", "error": str(e)})

    return result


# Check Ollama
ollama_health = check_endpoint_health("http://localhost:11434")
print("Ollama Health Check:")
print(json.dumps(ollama_health, indent=2))

# You can also check vLLM (if running)
print("\nvLLM Health Check (port 8000):")
vllm_health = check_endpoint_health("http://localhost:8000")
print(json.dumps(vllm_health, indent=2))

In [ ]:
# -------------------------------------------------------
# Inference Metrics Collector
# -------------------------------------------------------

class InferenceMetrics:
    """Collects and reports inference performance metrics."""

    def __init__(self):
        self.latencies = []
        self.token_counts = []
        self.errors = 0
        self.total_requests = 0

    def record(self, latency_ms: float, tokens: int, error: bool = False):
        self.total_requests += 1
        if error:
            self.errors += 1
        else:
            self.latencies.append(latency_ms)
            self.token_counts.append(tokens)

    def report(self) -> dict:
        if not self.latencies:
            return {"total_requests": self.total_requests, "errors": self.errors}

        sorted_lat = sorted(self.latencies)
        p50_idx = int(len(sorted_lat) * 0.50)
        p95_idx = int(len(sorted_lat) * 0.95)
        p99_idx = int(len(sorted_lat) * 0.99)

        total_tokens = sum(self.token_counts)
        total_time_s = sum(self.latencies) / 1000

        return {
            "total_requests": self.total_requests,
            "successful": len(self.latencies),
            "errors": self.errors,
            "error_rate_pct": round(self.errors / max(self.total_requests, 1) * 100, 1),
            "latency_p50_ms": round(sorted_lat[p50_idx], 1),
            "latency_p95_ms": round(sorted_lat[min(p95_idx, len(sorted_lat) - 1)], 1),
            "latency_p99_ms": round(sorted_lat[min(p99_idx, len(sorted_lat) - 1)], 1),
            "avg_latency_ms": round(statistics.mean(self.latencies), 1),
            "throughput_tok_per_s": round(total_tokens / max(total_time_s, 0.001), 1),
            "total_tokens": total_tokens,
        }


# Demo with synthetic data
metrics = InferenceMetrics()
import random
random.seed(42)
for _ in range(100):
    lat = random.gauss(250, 80)  # ~250ms mean latency
    tok = random.randint(50, 500)
    err = random.random() < 0.02  # 2% error rate
    metrics.record(max(lat, 10), tok, error=err)

report = metrics.report()
print("Inference Metrics Report (synthetic demo data):")
print("=" * 50)
for k, v in report.items():
    print(f"  {k}: {v}")

---
## Section 6: Latency & Throughput Benchmarking

Before choosing a hosting option, benchmark your actual workload.
The cell below sends multiple requests to an endpoint and measures
time-to-first-token (TTFT), total latency, and tokens per second.

In [ ]:
# -------------------------------------------------------
# Latency / Throughput Benchmarking
# -------------------------------------------------------

def benchmark_endpoint(
    base_url: str,
    model: str,
    prompt: str = "Explain the concept of PagedAttention in LLM serving in 3 sentences.",
    num_requests: int = 5,
    max_tokens: int = 200,
) -> dict:
    """
    Benchmark an OpenAI-compatible endpoint.
    Measures total latency and tokens/second for each request.
    """
    client = OpenAI(base_url=f"{base_url}/v1", api_key="benchmark")
    results = []

    print(f"Benchmarking {base_url} with model={model}, {num_requests} requests...")

    for i in range(num_requests):
        start = time.perf_counter()
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens,
                temperature=0.7,
            )
            elapsed_ms = (time.perf_counter() - start) * 1000
            total_tokens = response.usage.total_tokens if response.usage else 0
            completion_tokens = response.usage.completion_tokens if response.usage else 0
            tokens_per_sec = (completion_tokens / (elapsed_ms / 1000)) if elapsed_ms > 0 else 0

            results.append({
                "request": i + 1,
                "latency_ms": round(elapsed_ms, 1),
                "completion_tokens": completion_tokens,
                "tokens_per_sec": round(tokens_per_sec, 1),
                "error": False,
            })
            print(f"  [{i+1}/{num_requests}] {elapsed_ms:.0f}ms, "
                  f"{completion_tokens} tokens, {tokens_per_sec:.1f} tok/s")

        except Exception as e:
            elapsed_ms = (time.perf_counter() - start) * 1000
            results.append({
                "request": i + 1,
                "latency_ms": round(elapsed_ms, 1),
                "completion_tokens": 0,
                "tokens_per_sec": 0,
                "error": True,
            })
            print(f"  [{i+1}/{num_requests}] ERROR: {e}")

    # Aggregate
    successful = [r for r in results if not r["error"]]
    if successful:
        latencies = [r["latency_ms"] for r in successful]
        throughputs = [r["tokens_per_sec"] for r in successful]
        summary = {
            "endpoint": base_url,
            "model": model,
            "total_requests": num_requests,
            "successful": len(successful),
            "avg_latency_ms": round(statistics.mean(latencies), 1),
            "p50_latency_ms": round(statistics.median(latencies), 1),
            "avg_tok_per_sec": round(statistics.mean(throughputs), 1),
        }
    else:
        summary = {
            "endpoint": base_url,
            "model": model,
            "total_requests": num_requests,
            "successful": 0,
            "error": "All requests failed",
        }

    print(f"\nSummary: {json.dumps(summary, indent=2)}")
    return summary


# Benchmark Ollama (if available)
if available_models:
    ollama_bench = benchmark_endpoint(
        base_url="http://localhost:11434",
        model=MODEL,
        num_requests=5,
    )
else:
    print("Skipped: Ollama not available for benchmarking.")
    print("To benchmark vLLM, change base_url to 'http://localhost:8000'")

---
## Section 7: Docker Containerization

Docker packages your inference engine + API wrapper into a portable container.
The typical production setup uses a sidecar pattern:
- **Container 1**: vLLM engine on port 8000 (GPU inference)
- **Container 2**: FastAPI wrapper on port 8080 (validation, logging)

Below are the Dockerfile and startup script configurations.

In [ ]:
# -------------------------------------------------------
# Dockerfile for vLLM + FastAPI LLM Service
# -------------------------------------------------------

dockerfile_content = """# Dockerfile for LLM inference service
FROM vllm/vllm-openai:latest AS base

# Install additional dependencies for the API wrapper
RUN pip install fastapi uvicorn pydantic

# Copy application code
WORKDIR /app
COPY app/ /app/

# Expose ports: 8000 for vLLM, 8080 for FastAPI wrapper
EXPOSE 8000 8080

# Startup script that launches both vLLM and FastAPI
COPY start.sh /app/start.sh
RUN chmod +x /app/start.sh

ENTRYPOINT ["/app/start.sh"]
"""

startup_script = """#!/bin/bash
# start.sh -- Launch vLLM server and FastAPI wrapper

# Launch vLLM server in the background
vllm serve ${MODEL_ID:-meta-llama/Meta-Llama-3-8B-Instruct} \\
  --host 0.0.0.0 \\
  --port 8000 \\
  --dtype auto \\
  --max-model-len ${MAX_MODEL_LEN:-8192} \\
  --gpu-memory-utilization ${GPU_MEM_UTIL:-0.90} &

# Wait for vLLM to be ready
echo "Waiting for vLLM to start..."
until curl -s http://localhost:8000/health > /dev/null 2>&1; do
  sleep 2
done
echo "vLLM is ready!"

# Launch FastAPI wrapper
uvicorn app.main:app --host 0.0.0.0 --port 8080 --workers 2
"""

docker_compose = """# docker-compose.yml -- Local development stack
version: "3.8"
services:
  llm-service:
    build: .
    ports:
      - "8080:8080"
    environment:
      - MODEL_ID=meta-llama/Meta-Llama-3-8B-Instruct
      - MAX_MODEL_LEN=8192
      - GPU_MEM_UTIL=0.90
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: 1
              capabilities: [gpu]
    volumes:
      - model-cache:/root/.cache/huggingface
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8080/health"]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 600s

volumes:
  model-cache:
"""

print("=== Dockerfile ===")
print(dockerfile_content)
print("=== start.sh ===")
print(startup_script)
print("=== docker-compose.yml ===")
print(docker_compose)

In [ ]:
# -------------------------------------------------------
# AWS ECS Task Definition (GPU-enabled)
# -------------------------------------------------------

ecs_task_definition = {
    "family": "llm-inference",
    "requiresCompatibilities": ["EC2"],
    "networkMode": "awsvpc",
    "cpu": "8192",
    "memory": "30720",
    "containerDefinitions": [
        {
            "name": "llm-server",
            "image": "<ACCOUNT_ID>.dkr.ecr.us-east-1.amazonaws.com/llm-service:latest",
            "essential": True,
            "portMappings": [
                {"containerPort": 8080, "protocol": "tcp"}
            ],
            "resourceRequirements": [
                {"type": "GPU", "value": "1"}
            ],
            "environment": [
                {"name": "MODEL_ID", "value": "meta-llama/Meta-Llama-3-8B-Instruct"},
                {"name": "MAX_MODEL_LEN", "value": "8192"},
                {"name": "GPU_MEM_UTIL", "value": "0.90"},
            ],
            "mountPoints": [
                {
                    "sourceVolume": "model-cache",
                    "containerPath": "/root/.cache/huggingface",
                }
            ],
            "healthCheck": {
                "command": ["CMD-SHELL", "curl -f http://localhost:8080/health || exit 1"],
                "interval": 30,
                "timeout": 10,
                "retries": 3,
                "startPeriod": 600,
            },
            "logConfiguration": {
                "logDriver": "awslogs",
                "options": {
                    "awslogs-group": "/ecs/llm-inference",
                    "awslogs-region": "us-east-1",
                    "awslogs-stream-prefix": "llm",
                },
            },
        }
    ],
    "volumes": [
        {
            "name": "model-cache",
            "efsVolumeConfiguration": {
                "fileSystemId": "fs-0123456789abcdef0",
                "rootDirectory": "/models",
            },
        }
    ],
}

print("ECS Task Definition (GPU-enabled):")
print("=" * 50)
print(json.dumps(ecs_task_definition, indent=2))

---
## Section 8: Cloud Deployment -- SageMaker & Instance Selection

AWS SageMaker provides fully managed LLM endpoints. You hand over model weights
and configuration; AWS handles GPU provisioning, load balancing, auto-scaling,
and health checks.

### GPU Instance Selection Guide

In [ ]:
# -------------------------------------------------------
# GPU Instance Type Reference & Cost Calculator
# -------------------------------------------------------

gpu_instances = [
    {
        "instance": "ml.g5.xlarge",
        "gpu": "1x A10G",
        "vram_gb": 24,
        "best_for": "7B models (quantized)",
        "cost_per_hr": 1.41,
    },
    {
        "instance": "ml.g5.2xlarge",
        "gpu": "1x A10G",
        "vram_gb": 24,
        "best_for": "7-8B models (FP16)",
        "cost_per_hr": 1.52,
    },
    {
        "instance": "ml.g5.12xlarge",
        "gpu": "4x A10G",
        "vram_gb": 96,
        "best_for": "70B models (quantized)",
        "cost_per_hr": 7.09,
    },
    {
        "instance": "ml.g6.xlarge",
        "gpu": "1x L4",
        "vram_gb": 24,
        "best_for": "7B models (cost-optimized)",
        "cost_per_hr": 0.98,
    },
    {
        "instance": "ml.p4d.24xlarge",
        "gpu": "8x A100",
        "vram_gb": 640,
        "best_for": "70B+ models (FP16)",
        "cost_per_hr": 42.60,
    },
]

table_data = []
for inst in gpu_instances:
    monthly_24x7 = inst["cost_per_hr"] * 24 * 30
    monthly_biz = inst["cost_per_hr"] * 10 * 22  # 10hrs/day, 22 days/month
    table_data.append([
        inst["instance"],
        inst["gpu"],
        f"{inst['vram_gb']} GB",
        inst["best_for"],
        f"${inst['cost_per_hr']:.2f}",
        f"${monthly_24x7:,.0f}",
        f"${monthly_biz:,.0f}",
    ])

print("SageMaker GPU Instance Reference")
print("=" * 100)
print(
    tabulate(
        table_data,
        headers=[
            "Instance", "GPU", "VRAM", "Best For",
            "$/hr", "$/mo (24x7)", "$/mo (biz hrs)",
        ],
        tablefmt="grid",
    )
)
print("\nTip: Always implement auto-scaling. A single g5.2xlarge running 24/7 = ~$1,100/mo.")

In [ ]:
# -------------------------------------------------------
# SageMaker Deployment Example (reference code)
# -------------------------------------------------------
# This code deploys a Llama 3 8B model to a SageMaker real-time endpoint.
# It requires AWS credentials and will incur costs -- do NOT run unless intended.

SAGEMAKER_DEPLOY_CODE = '''
import sagemaker
from sagemaker.huggingface import HuggingFaceModel, get_huggingface_llm_image_uri

role = sagemaker.get_execution_role()
sess = sagemaker.Session()

# Get the latest TGI container image URI
image_uri = get_huggingface_llm_image_uri(
    backend="huggingface",   # "huggingface" = TGI, or use "lmi" for vLLM
    region=sess.boto_region_name,
    version="2.3.1"
)

# Define the model
model = HuggingFaceModel(
    role=role,
    image_uri=image_uri,
    env={
        "HF_MODEL_ID": "meta-llama/Meta-Llama-3-8B-Instruct",
        "HF_TOKEN": os.environ["HF_TOKEN"],  # For gated models
        "SM_NUM_GPUS": "1",
        "MAX_INPUT_TOKENS": "4096",
        "MAX_TOTAL_TOKENS": "8192",
        "MAX_BATCH_PREFILL_TOKENS": "4096",
    }
)

# Deploy to a real-time endpoint
predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.g5.2xlarge",  # 1x A10G 24GB
    endpoint_name="llama3-8b-endpoint",
    container_startup_health_check_timeout=600,
)

print(f"Endpoint ready: {predictor.endpoint_name}")
'''

print("SageMaker Deployment Code (reference -- do not run without AWS credentials):")
print("=" * 70)
print(SAGEMAKER_DEPLOY_CODE)

---
## Section 9: Comprehensive Cost Comparison

This section brings together all hosting options and compares costs, latency, and
operational complexity side by side.

In [ ]:
# -------------------------------------------------------
# Full Hosting Option Comparison
# -------------------------------------------------------

hosting_options = [
    {
        "option": "OpenAI API (GPT-4o)",
        "setup_time": "Minutes",
        "monthly_cost_10k_qpd": "$3,000",
        "latency_ttft": "300-800ms",
        "scaling": "Automatic",
        "data_privacy": "Third-party",
        "ops_burden": "None",
        "model_choice": "OpenAI models only",
    },
    {
        "option": "Ollama (Local Dev)",
        "setup_time": "Minutes",
        "monthly_cost_10k_qpd": "$0 (your hardware)",
        "latency_ttft": "50-200ms",
        "scaling": "Single machine",
        "data_privacy": "Full control",
        "ops_burden": "Minimal",
        "model_choice": "Any GGUF model",
    },
    {
        "option": "vLLM on EC2/ECS",
        "setup_time": "Hours",
        "monthly_cost_10k_qpd": "$1,100-$1,500",
        "latency_ttft": "30-100ms",
        "scaling": "Manual / ASG",
        "data_privacy": "Full control (VPC)",
        "ops_burden": "High",
        "model_choice": "Any HF model",
    },
    {
        "option": "SageMaker Endpoint",
        "setup_time": "30 min",
        "monthly_cost_10k_qpd": "$1,100-$2,000",
        "latency_ttft": "50-200ms",
        "scaling": "Auto-scaling",
        "data_privacy": "AWS VPC",
        "ops_burden": "Medium",
        "model_choice": "Any HF model",
    },
    {
        "option": "TGI on Docker",
        "setup_time": "Hours",
        "monthly_cost_10k_qpd": "$1,100-$1,500",
        "latency_ttft": "30-100ms",
        "scaling": "Manual / K8s",
        "data_privacy": "Full control",
        "ops_burden": "High",
        "model_choice": "Any HF model",
    },
]

table_data = [
    [
        opt["option"],
        opt["setup_time"],
        opt["monthly_cost_10k_qpd"],
        opt["latency_ttft"],
        opt["scaling"],
        opt["data_privacy"],
        opt["ops_burden"],
    ]
    for opt in hosting_options
]

print("LLM Hosting Options -- Full Comparison (at 10K queries/day)")
print("=" * 110)
print(
    tabulate(
        table_data,
        headers=["Option", "Setup", "~Monthly Cost", "TTFT", "Scaling", "Privacy", "Ops"],
        tablefmt="grid",
    )
)

In [ ]:
# -------------------------------------------------------
# Cost Crossover Analysis: At What Volume Does Self-Hosting Win?
# -------------------------------------------------------

print("Cost Crossover Analysis")
print("=" * 70)
print("Comparing GPT-4o API vs. self-hosted Llama 3 8B on g5.2xlarge\n")

# Fixed self-hosted monthly cost
gpu_monthly = 1.52 * 24 * 30  # g5.2xlarge 24/7
eng_monthly = 20 * 100         # 20 hrs @ $100/hr
self_hosted_fixed = gpu_monthly + eng_monthly

print(f"Self-hosted fixed monthly cost: ${self_hosted_fixed:,.0f}")
print(f"  GPU (g5.2xlarge 24/7):  ${gpu_monthly:,.0f}")
print(f"  Engineering (20 hrs):   ${eng_monthly:,.0f}")
print()

# Find crossover point
# API cost per query = (2000 * 2.50 + 500 * 10.00) / 1_000_000
api_cost_per_query = (2000 * 2.50 + 500 * 10.00) / 1_000_000
print(f"API cost per query: ${api_cost_per_query:.4f}")

# Crossover: api_cost_per_query * queries_per_month = self_hosted_fixed
crossover_monthly = self_hosted_fixed / api_cost_per_query
crossover_daily = crossover_monthly / 30

print(f"\nCrossover point: {crossover_daily:,.0f} queries/day ({crossover_monthly:,.0f}/month)")
print(f"Below this volume -> Use API")
print(f"Above this volume -> Self-host")

# Show savings at various volumes
print(f"\nSavings at different volumes:")
for qpd in [1000, 5000, 10000, 50000, 100000, 500000]:
    api_cost = qpd * 30 * api_cost_per_query
    savings = api_cost - self_hosted_fixed
    winner = "Self-Host" if savings > 0 else "API"
    print(f"  {qpd:>7,} queries/day: API=${api_cost:>10,.0f}  "
          f"Self=${self_hosted_fixed:>7,.0f}  "
          f"Savings=${savings:>+10,.0f}  -> {winner}")

---
## Summary

In this notebook we covered:

1. **Self-hosting decision framework** -- Cost, latency, privacy, customization, and operational maturity dimensions
2. **Inference engines** -- vLLM (PagedAttention, OpenAI-compatible) vs. TGI (Rust core, HF ecosystem) vs. Ollama (local dev)
3. **Ollama for local development** -- Native API, streaming, OpenAI SDK compatibility, and Modelfiles
4. **FastAPI wrapper services** -- Pydantic validation, streaming SSE, health checks, and error handling
5. **Health checks & monitoring** -- Liveness/readiness probes, latency percentiles, throughput metrics
6. **Benchmarking** -- Measuring TTFT, tokens/second, and comparing endpoints
7. **Docker containerization** -- Dockerfile, startup scripts, docker-compose, and ECS task definitions
8. **Cloud deployment** -- SageMaker endpoints, GPU instance selection, and auto-scaling
9. **Cost analysis** -- Crossover calculations showing when self-hosting becomes economical

### Key Takeaways
- Start with commercial APIs; move to self-hosting when cost, privacy, or latency demands it
- Use Ollama for development, vLLM for production throughput
- The OpenAI-compatible API pattern lets you swap backends by changing a single URL
- Always implement health checks, monitoring, and auto-scaling for production deployments

**Next module**: `06-prompt-engineering.html` -- Prompt engineering techniques and optimization.